# Anti-spoofing: train and compare MobileNetV3 vs DeepPixBiS

**Owner:** Hoan · **Task:** antispoofing · **Models:** MobileNetV3-Large (binary) and DeepPixBiS (pixel-wise supervision)

Run top to bottom on a Colab GPU. All logic lives in `src/antispoofing/`; this notebook only calls it,
so the same commands work from a terminal.

**References:** A. George and S. Marcel, *Deep Pixel-wise Binary Supervision for Face Presentation Attack Detection*, ICB 2019 ·
A. Howard et al., *Searching for MobileNetV3*, ICCV 2019 · Y. Zhang et al., *CelebA-Spoof*, ECCV 2020 ·
ISO/IEC 30107-3 (APCER / BPCER / ACER).

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/henryhoang207/Facial-Recognition---COS30082.git"
BRANCH = "main"  # change to your feature branch if needed

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    REPO_DIR = Path("/content/repo" if IN_COLAB else "/kaggle/working/repo")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    # torch / torchvision are preinstalled; install the rest only
    reqs = [l for l in (REPO_DIR / "requirements.txt").read_text().splitlines()
            if l and not l.startswith("#") and not l.startswith("torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO_DIR))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/FaceGuard")
elif IN_KAGGLE:
    DRIVE_DIR = Path("/kaggle/working/FaceGuard")  # add data as a Kaggle dataset; download checkpoints at the end
else:
    DRIVE_DIR = REPO_DIR

DATA_DIR = DRIVE_DIR / "data"
CKPT_DIR = DRIVE_DIR / "checkpoints"
FIG_DIR = REPO_DIR / "reports" / "figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR, "| data:", DATA_DIR, "| checkpoints:", CKPT_DIR)

## 2. Data to local disk
Reading many small images straight from Drive is slow. Copy the zips once per session and unzip to `/content/data`.
Expected on Drive (Thành's layout, see `data/README.md`):
- `FaceGuard/data/spoof_public.zip` → `spoof_public/CelebA_Spoof/{Data,metas}/...`
- `FaceGuard/data/spoof_recorded.zip` → `spoof_recorded/<person>/<live|print|replay>/<images or videos>`

In [ ]:
LOCAL_DATA = Path("/content/data") if IN_COLAB else DATA_DIR
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
for name in ["spoof_public", "spoof_recorded"]:
    zip_path = DATA_DIR / f"{name}.zip"
    if (LOCAL_DATA / name).exists():
        print("already here:", name)
    elif zip_path.exists():
        subprocess.run(["unzip", "-q", str(zip_path), "-d", str(LOCAL_DATA)], check=True)
        print("unzipped", name)
    else:
        print("missing", zip_path)

## 3. Build manifests
One CSV per source. Splits are by person, so nobody is in two splits.
CelebA-Spoof: print and replay attacks only (what our demo faces), up to 20 images per person per class.
Recorded: put the people listed in `TEST_PEOPLE` in test only.

In [ ]:
MANIFEST_DIR = DRIVE_DIR / "data" / "antispoofing"
CELEBA = MANIFEST_DIR / "celeba.csv"
RECORDED = MANIFEST_DIR / "recorded.csv"
TEST_PEOPLE = "person1,person2"  # recorded people kept out of training; at least 2

%cd {REPO_DIR}
if (LOCAL_DATA / "spoof_public").exists():
    !python -m src.antispoofing.manifest celeba --data-root {LOCAL_DATA} --root {LOCAL_DATA}/spoof_public/CelebA_Spoof --attacks print,replay --max-per-class 20 --out {CELEBA}
if (LOCAL_DATA / "spoof_recorded").exists():
    !python -m src.antispoofing.manifest folder --data-root {LOCAL_DATA} --root {LOCAL_DATA}/spoof_recorded --source recorded --test-subjects {TEST_PEOPLE} --out {RECORDED}
MANIFESTS = " ".join(str(p) for p in [CELEBA, RECORDED] if p.exists())
print(MANIFESTS)

### Look at the crops
Each crop should show the face in the middle with a margin around it. If faces are off-centre or cut off, the box scale is wrong: stop and fix `read_celeba_bbox`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from src.common.face_crop import crop_face

sample = pd.concat([pd.read_csv(p) for p in MANIFESTS.split()]).groupby(["source", "attack"]).head(2)
fig, axes = plt.subplots(1, len(sample), figsize=(2 * len(sample), 2.4))
for ax, r in zip(axes, sample.itertuples()):
    face = crop_face(cv2.imread(str(LOCAL_DATA / r.path)), (r.x1, r.y1, r.x2, r.y2), size=224, scale=2.0)
    ax.imshow(face[:, :, ::-1]); ax.set_title(f"{r.source}\n{r.attack}", fontsize=8); ax.axis("off")

## 4. Quick check (2 minutes)
One short epoch on 2,000 images. If this fails, fix it before the long runs.

In [ ]:
!python -m src.antispoofing.train --model mobilenetv3 --manifest {MANIFESTS} --data-root {LOCAL_DATA} --ckpt-dir /content/tmp_ckpt --fig-dir /content/tmp_fig --tag smoke --epochs 1 --max-train 2000

## 5. Train both candidates
Same data, same crop (2.0 × face box, 224 px), same epochs and early stopping. Only the model and its loss differ.
The best epoch by val ACER is saved to `FaceGuard/checkpoints/antispoofing_<model>_<date>.pt` with its val threshold.

In [ ]:
EPOCHS = 15
!python -m src.antispoofing.train --model mobilenetv3 --manifest {MANIFESTS} --data-root {LOCAL_DATA} --ckpt-dir {CKPT_DIR} --epochs {EPOCHS}

In [ ]:
!python -m src.antispoofing.train --model deeppixbis --manifest {MANIFESTS} --data-root {LOCAL_DATA} --ckpt-dir {CKPT_DIR} --epochs {EPOCHS}

## 6. Evaluate on test (people never seen in training)
Uses each checkpoint's val threshold. Rows are added to `reports/figures/antispoofing_results.csv`.

In [ ]:
for model in ["mobilenetv3", "deeppixbis"]:
    ckpt = sorted(CKPT_DIR.glob(f"antispoofing_{model}_*.pt"))[-1]  # newest
    !python -m src.antispoofing.evaluate --checkpoint {ckpt} --manifest {MANIFESTS} --data-root {LOCAL_DATA}

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((FIG_DIR / "antispoofing_results.md").read_text()))
for model in ["mobilenetv3", "deeppixbis"]:
    for kind in ["training", "roc", "scores"]:
        path = FIG_DIR / f"antispoofing_{model}_{kind}.png"
        if path.exists():
            display(Image(str(path)))

## 7. Save results to Drive
The repo clone on Colab is deleted when the session ends. Copy the figures to Drive, then commit them from your PC.

In [ ]:
import shutil
dest = DRIVE_DIR / "results" / "antispoofing"
dest.mkdir(parents=True, exist_ok=True)
for f in FIG_DIR.glob("antispoofing_*"):
    shutil.copy(f, dest)
print("copied to", dest)

## 8. Results and notes
Check against the targets in the README (Anti-spoofing section). Then 3–5 lines: which model wins on the recorded test people,
which attack is hardest (see APCER per attack), and what to try next.